# trade-ngin Python Wrapper — End-to-End Pipeline

This notebook walks through the full backtest pipeline exposed by the `tradengin` pybind11
wrapper (`feature/wrapper` branch): building the module, connecting configuration, defining a
strategy in Python, and running a backtest.

## What the wrapper actually exposes

The wrapper is a thin surface over the C++ engine. Only these symbols are bound
(see `python/module.cpp`):

| Python name | C++ source | Purpose |
|---|---|---|
| `Bar`, `Position`, `ExecutionReport` | `core/types.hpp` | Market data + position records |
| `BacktestResults` | `backtest/backtest_types.hpp` | Performance metrics returned by a run |
| `StrategyConfig` | `strategy/types.hpp` | Capital, leverage, limits, costs |
| `PostgresDatabase` | `data/postgres_database.hpp` | DB handle (connect/disconnect only) |
| `Instrument`, `InstrumentRegistry` | `instruments/` | Contract metadata lookup |
| `BaseStrategy` | `strategy/base_strategy.hpp` | Subclass this to write a strategy in Python |
| `BacktestRunner` | `api/backtest_api.hpp` | Orchestrates the whole run |

**Not bound:** `TradeError` / `Result` are commented out in `type_bindings.cpp`. C++ errors surface
as plain Python `RuntimeError` with the message only — the error code and component are lost.
That is a known limitation, discussed at the end.

---
## 1. Build and install the module

The wrapper is **not** on PyPI in a usable prebuilt form for local development — it compiles the
entire C++ engine, which needs native dependencies first.

### 1a. Native dependencies (vcpkg)

`vcpkg.json` requires: `arrow`, `curl`, `eigen3`, `libpqxx`, `nlopt`, `nlohmann-json`, `gtest`.
Arrow is large; expect a long first build.

```bash
git clone https://github.com/microsoft/vcpkg.git
./vcpkg/bootstrap-vcpkg.sh          # .bat on Windows

git clone --branch feature/wrapper https://github.com/AlgoGators/trade-ngin.git
cd trade-ngin
git submodule update --init --recursive   # pulls extern/pybind11 (v3.0.3)

../vcpkg/vcpkg install --triplet x64-windows   # x64-linux / arm64-osx elsewhere
```

### 1b. Build the wheel

`pyproject.toml` uses `scikit-build-core`, so a normal pip install drives CMake:

```bash
export VCPKG_ROOT=/path/to/vcpkg   # presets and pip both read this
pip install . \
  --config-settings=cmake.args="-DCMAKE_TOOLCHAIN_FILE=$VCPKG_ROOT/scripts/buildsystems/vcpkg.cmake"
```

`BUILD_PYTHON_WRAPPER` defaults to `ON` (root `CMakeLists.txt`), so the module is built
automatically. The compiled artifact is named `tradengin` (set via `OUTPUT_NAME` in
`python/CMakeLists.txt`) even though the distribution is named `trade-ngin`.

> **Import name vs package name:** `pip install trade-ngin` but `import tradengin`.

In [ ]:
# Verify the module imports and inspect what it exposes.
# If this raises ImportError, the build step above did not complete.
import tradengin

print("module file:", tradengin.__file__)
print("exported symbols:")
for name in sorted(n for n in dir(tradengin) if not n.startswith("_")):
    print("   ", name)

---
## 2. Configuration

`BacktestRunner` reads config from **disk**, not from Python arguments. It calls
`ConfigLoader::load("config", portfolio_name)`, which expects this layout relative to the
**current working directory**:

```
config/
├── defaults.json                 # DB credentials, execution, optimization, backtest window
└── portfolios/
    └── <portfolio_name>/
        ├── portfolio.json        # portfolio_id, capital, strategy list + allocations
        ├── risk.json
        └── email.json
```

Copy `config_template/` to `config/` and fill in the placeholders. The key fields:

- `defaults.json → database`: real Postgres host/user/password/name. **A live database is
  mandatory** — the run loads instruments and price history from it.
- `defaults.json → backtest.lookback_years`: how far back the run starts (end date is *today*).
- `portfolio.json → strategies`: each entry needs `enabled_backtest: true` and a
  `default_allocation`. Allocations are normalized to sum to 1.0.

**A strategy in `portfolio.json` must also be registered from Python by the exact same key**
(e.g. `TREND_FOLLOWING`), or the run aborts with `Strategy <id> is not registered`.

In [ ]:
# Sanity-check the config tree before running anything, since a missing file surfaces
# as a generic RuntimeError deep inside the C++ call.
from pathlib import Path
import json

# "conservative" enables only TREND_FOLLOWING, so the single registration in section 3
# satisfies it. "base" also enables TREND_FOLLOWING_FAST and would abort unless that ID
# is registered too -- see section 7.
PORTFOLIO = "conservative"  # matches config/portfolios/<name>/
cfg_root = Path("config")

required = [
    cfg_root / "defaults.json",
    cfg_root / "portfolios" / PORTFOLIO / "portfolio.json",
    cfg_root / "portfolios" / PORTFOLIO / "risk.json",
]
for path in required:
    print(f"{'OK  ' if path.exists() else 'MISS'} {path}")

# Show which strategy IDs the backtest will expect to be registered.
pf_file = cfg_root / "portfolios" / PORTFOLIO / "portfolio.json"
if pf_file.exists():
    pf = json.loads(pf_file.read_text())
    enabled = [k for k, v in pf.get("strategies", {}).items() if v.get("enabled_backtest")]
    print("\nportfolio_id:", pf.get("portfolio_id"))
    print("initial_capital:", pf.get("initial_capital"))
    print("strategies needing registration:", enabled)

---
## 3. The fast path — built-in trend following

The wrapper ships a convenience binding that registers the native C++ trend-following strategy
under the ID `TREND_FOLLOWING`. Nothing is executed in Python, so this is the fastest route and a
good first smoke test that your DB and config are wired correctly.

In [ ]:
runner = tradengin.BacktestRunner()

# initialize() sets the portfolio name and configures the C++ logger
# (writes to ./logs/bt_portfolio_<name>*). It does NOT touch the database yet.
runner.initialize(PORTFOLIO)

# Registers the native TrendFollowingStrategy under the hard-coded ID "TREND_FOLLOWING".
# Its parameters are read from portfolio.json -> strategies.TREND_FOLLOWING.config.
runner.register_trend_following_strategy()

print("registered; ready to run")

In [ ]:
# run_backtest() performs the entire pipeline in one blocking C++ call:
#   load config -> open DB pool -> load instruments -> fetch symbols & bars
#   -> build strategies -> run portfolio -> persist results to DB -> return metrics
#
# On failure the C++ Result<T> is converted to a Python exception by the custom caster in
# python/bindings/result_caster.hpp, so failures arrive as RuntimeError.
try:
    results = runner.run_backtest()
except RuntimeError as exc:
    print("backtest failed:", exc)
    results = None

if results is not None:
    print(results)  # __str__ prints the full metric set

In [ ]:
# BacktestResults exposes each metric as a readable/writable attribute.
if results is not None:
    metrics = {
        "total_return":        results.total_return,
        "volatility":          results.volatility,
        "sharpe_ratio":        results.sharpe_ratio,
        "sortino_ratio":       results.sortino_ratio,
        "max_drawdown":        results.max_drawdown,
        "calmar_ratio":        results.calmar_ratio,
        "total_trades":        results.total_trades,
        "win_rate":            results.win_rate,
        "profit_factor":       results.profit_factor,
        "var_95":              results.var_95,
        "cvar_95":             results.cvar_95,
        "downside_volatility": results.downside_volatility,
    }
    width = max(len(k) for k in metrics)
    for key, value in metrics.items():
        print(f"{key:<{width}}  {value}")

---
## 4. Writing a strategy in Python

This is the point of the wrapper. Subclass `tradengin.BaseStrategy` and override
`generate_positions_from_data`. The C++ trampoline (`PyBaseStrategy` in
`python/bindings/strategy_bindings.cpp`) routes the engine's `on_data` callback into your Python
method, then feeds whatever positions you return back into the portfolio via `update_position`.

### Contract you must follow

- **`generate_positions_from_data(self, bars) -> list[Position]`** — called once per bar batch.
  Return a list of `Position` objects. Returning `None` is treated as "no positions".
- **`initialize(self)`** *(optional)* — called once at startup. Return `None` for success.
  The base C++ initialization always runs first regardless.
- **`get_price_history(self)`** *(optional)* — return `dict[str, list[float]]`. If you do not
  override it, the C++ base implementation is used.

### Two hard constraints

1. **Do not define `__init__` with required arguments.** The factory instantiates your class with
   `py_class()` — no arguments. If you need setup, do it in `initialize()`.
2. **Exceptions raised inside your overrides are swallowed.** The trampoline catches
   `py::error_already_set`, logs it, and returns an empty position list. A crashing strategy looks
   identical to a flat one, so guard your own code and log explicitly.

In [ ]:
from tradengin import BaseStrategy, Position


class MomentumStrategy(BaseStrategy):
    """Toy long-only momentum strategy: hold 1 contract when the short SMA exceeds the long SMA.

    Illustrates the required shape of a Python strategy, not a viable trading model.
    """

    # NOTE: no __init__ — the factory calls MomentumStrategy() with no arguments.

    SHORT_WINDOW = 16
    LONG_WINDOW = 64

    def initialize(self):
        # Runs once, after the C++ base class has been initialized and the context
        # (id, config, db, registry) has been injected. Returning None signals success.
        # Engine-driven runs land here. Manual/unit-test drives may skip
        # initialize() entirely, so state is (re)created lazily below too.
        self.closes = {}
        print(f"[MomentumStrategy] initialized, capital={self.config.capital_allocation}")
        return None

    def generate_positions_from_data(self, bars):
        # `bars` is a list of tradengin.Bar for the current step.
        # Wrap the body so an exception cannot silently become "no positions".
        try:
            positions = []

            for bar in bars:
                # Lazily create state: initialize() is not called when a bare
                # instance is driven by hand (see section 6).
                if not hasattr(self, "closes"):
                    self.closes = {}
                history = self.closes.setdefault(bar.symbol, [])
                history.append(float(bar.close))
                # Keep only what the longest window needs.
                if len(history) > self.LONG_WINDOW:
                    del history[0]

                if len(history) < self.LONG_WINDOW:
                    continue  # not enough data to form a signal yet

                short_ma = sum(history[-self.SHORT_WINDOW:]) / self.SHORT_WINDOW
                long_ma = sum(history) / self.LONG_WINDOW

                pos = Position()
                pos.symbol = bar.symbol
                pos.quantity = 1 if short_ma > long_ma else 0
                pos.avg_price = bar.close
                pos.last_update = bar.timestamp
                positions.append(pos)

            return positions

        except Exception as exc:
            # The trampoline would otherwise hide this entirely.
            print(f"[MomentumStrategy] ERROR: {exc!r}")
            return []


print("strategy class defined")

In [ ]:
# Register the Python class against a strategy ID that exists in portfolio.json.
# Pass the CLASS itself, not an instance -- the C++ factory instantiates it per run.
py_runner = tradengin.BacktestRunner()
py_runner.initialize(PORTFOLIO)

# The ID must match a key under "strategies" in portfolio.json with enabled_backtest=true.
py_runner.register_strategy("TREND_FOLLOWING", MomentumStrategy)

try:
    py_results = py_runner.run_backtest()
    print(py_results)
except RuntimeError as exc:
    print("backtest failed:", exc)
    py_results = None

---
## 5. Inspecting instruments

`InstrumentRegistry` is a singleton populated from the database during `run_backtest()`. Use it to
look up contract metadata — most usefully `get_multiplier()`, which converts a price move into a
cash P&L and is required for correct position sizing.

> The registry is only populated **after** a backtest has run. Querying it beforehand returns
> nothing.

In [ ]:
registry = tradengin.InstrumentRegistry.instance()

for symbol in ["ES.v.0", "ZN.v.0", "CL.v.0"]:
    if registry.has_instrument(symbol):
        inst = registry.get_instrument(symbol)
        print(
            f"{inst.get_symbol():<10} "
            f"multiplier={inst.get_multiplier():<10} "
            f"tick={inst.get_tick_size():<10} "
            f"point_value={inst.get_point_value()}"
        )
    else:
        print(f"{symbol:<10} not in registry")

---
## 6. Constructing types directly

`Bar` and `Position` are plain bindings with default constructors and writable fields, so they can
be built in Python for unit-testing a strategy without a database.

Note the field renames between C++ and Python: `Position::average_price` is exposed as
`avg_price`, and `ExecutionReport::filled_quantity` / `fill_price` / `fill_time` become
`quantity` / `price` / `timestamp`.

In [ ]:
from datetime import datetime, timedelta

# Build a synthetic rising series and push it through the strategy in isolation.
start = datetime(2024, 1, 1)
synthetic = []
for i in range(80):
    bar = tradengin.Bar()
    bar.symbol = "TEST.v.0"
    bar.timestamp = start + timedelta(days=i)
    bar.open = bar.high = bar.low = bar.close = 100.0 + i  # steadily trending up
    bar.volume = 1000.0
    synthetic.append(bar)

# Drive the strategy by hand -- no engine, no database.
#
# NOTE: do NOT call initialize() here. It chains into BaseStrategy::initialize(),
# which requires the engine to have injected context first (it checks
# state_ == INITIALIZED and a non-null db_) and otherwise raises
# RuntimeError("Strategy not initialized"). The strategy above creates its
# state lazily so it works either way.
strat = MomentumStrategy()


out = strat.generate_positions_from_data(synthetic)
print(f"returned {len(out)} position(s)")
for p in out:
    print(f"  {p.symbol} qty={p.quantity} avg_price={p.avg_price}")

---
## 7. Known limitations and gotchas

Findings from reading the wrapper source — worth knowing before relying on it.

### Error reporting is lossy
`TradeError` and `Result` bindings are commented out in `type_bindings.cpp`. The caster throws a
plain `std::runtime_error` built from the `TradeError` base subobject, so the **error code and
component are discarded**. You get the message string and nothing else — catch `RuntimeError` and
parse text, or read `logs/` for the full context.

### Silent failures inside Python overrides
As noted in §4, exceptions in `generate_positions_from_data` and `get_price_history` are caught
and converted to empty results. Always wrap your logic in `try/except` and log.

### `fdm` cannot be overridden from config
In `api_bindings.cpp`, `register_trend_following_strategy` guards the FDM assignment with
`if (trend_config.fdm.empty())`. `TrendFollowingConfig::fdm` has a **non-empty default**, so that
branch never fires and any `fdm` supplied in `portfolio.json` is ignored.

### One trend-following registration only
`register_trend_following_strategy()` hard-codes the ID `TREND_FOLLOWING`. A portfolio that also
defines `TREND_FOLLOWING_FAST` (as `config_template/portfolios/base` does) will abort with
`Strategy TREND_FOLLOWING_FAST is not registered` unless you register that ID yourself.

### Latent bug in the `Result<T>` caster
`result_caster.hpp` assigns `Result<void>()` into a `Result<T>` slot in its `load()` path. It
compiles today only because the sole load site uses `Result<void>`, which hits the specialization.
Binding any function that *accepts* a `Result<T>` parameter will break the build.

### A live database is required
There is no offline or CSV mode in this API. `run_backtest()` needs Postgres reachable with
instrument and price data loaded, and it **writes results back** to the database on completion.

### Results are not returned as a DataFrame
Only the scalar metrics on `BacktestResults` are bound — equity curves, per-trade records, and
per-strategy breakdowns are not exposed to Python (`# TODO add bindings for other results` in
`type_bindings.cpp`). Query the results tables in Postgres for that detail.